# ⚡ 玻璃短线（小资金 1 万）

> 基于 **vnpy 4.4 · `vnpy.alpha`**，玻璃主力连续 `FG00.CZCE`，**起始资金 10,000 元**。

| 环节 | 内容 |
|---|---|
| 📈 数据 | 日线行情可视化 |
| 📐 信号 | 三均线趋势 **3/21/34**（日线级短线波段） |
| 💰 仓位 | 按 **14% 保证金** 估算，最多 **1 手** |
| 💵 手续费 | 开/平各 **2 元/手**，往返 **4 元/手** |
| 🏆 回测 | TEST 样本外 2023-07 ~ 2026-06，约 **+293%** |

**说明**：仓库仅有日线数据，「短线」指约 3~10 日持仓的波段，非分钟级。

## 0️⃣ 环境与配置

In [ ]:
import sys, os, warnings, importlib
from pathlib import Path
warnings.filterwarnings("ignore")

# 定位 fg_short 目录（支持从 fg_short/ 或仓库根打开 notebook）
_cwd = Path.cwd()
FG_DIR = _cwd if (_cwd / "config.py").exists() else _cwd / "fg_short"
if not (FG_DIR / "config.py").exists():
    raise FileNotFoundError(f"找不到 config.py，请 cd 到 fg_short 再打开 notebook。当前: {_cwd}")
if str(FG_DIR) not in sys.path:
    sys.path.insert(0, str(FG_DIR))
os.chdir(FG_DIR)
os.environ.setdefault("OMP_NUM_THREADS", "2")

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 110

# 强制从 fg_short/config.py 加载（避免误导入其他包的 config）
import importlib.util
for _mod in list(sys.modules):
    if _mod in ("config", "data", "signals", "strategy", "backtest") or _mod.startswith("config."):
        del sys.modules[_mod]
_cfg_path = FG_DIR / "config.py"
_spec = importlib.util.spec_from_file_location("fg_short_config", _cfg_path)
config = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(config)
sys.modules["config"] = config

import data
from signals import generate_signals
from strategy import ShortTermStrategy

config.ensure_dirs()
print("工作目录:", FG_DIR)
print("配置就绪 ✅  合约:", config.VT_SYMBOL)
print(f"  三均线: {config.MA_SHORT}/{config.MA_MEDIUM}/{config.MA_LONG}  "
      f"资金={config.CAPITAL:,}  最多{config.MAX_LOTS}手  保证金率={config.MARGIN_RATE:.0%}")
print(f"  手续费: 开/平各 {config.COMMISSION_YUAN_PER_LOT:.0f} 元/手  "
      f"费率≈{config.LONG_RATE:.5f} (参考价 {config.COMMISSION_REF_PRICE:.0f})")

## 1️⃣ 数据加载与行情可视化 📈

In [ ]:
bars_df = data.load_polars_df()
pdf = bars_df.to_pandas()
pdf["datetime"] = pd.to_datetime(pdf["datetime"])
print(f"共 {len(pdf)} 根日线  {pdf['datetime'].min().date()} ~ {pdf['datetime'].max().date()}")
print(f"最新收盘: {pdf['close'].iloc[-1]:.0f}  1手名义≈{pdf['close'].iloc[-1]*config.CONTRACT_SIZE:,.0f} 元")
print(f"1手保证金(14%)≈{pdf['close'].iloc[-1]*config.CONTRACT_SIZE*config.MARGIN_RATE:,.0f} 元")

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
ax = axes[0]
ax.plot(pdf["datetime"], pdf["close"], color="#2563eb", lw=1, label="收盘价")
ax.set_title(f"{config.VT_SYMBOL} 玻璃主力连续 — 收盘价")
ax.set_ylabel("价格")
ax.legend(loc="upper left")
ax.grid(alpha=0.3)

ax2 = axes[1]
ax2.bar(pdf["datetime"], pdf["volume"], color="#94a3b8", width=1.0, alpha=0.7)
ax2.set_ylabel("成交量")
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 2️⃣ 三均线信号 📐

规则（与 `fg_ai` 三均线战法一致，周期更短）：

- **做多**：短 > 中 > 长，且收盘 > 中均线
- **做空**：短 < 中 < 长，且收盘 < 中均线
- **空仓**：均线缠绕

In [ ]:
signal_df = generate_signals(
    bars_df,
    mode=config.SIGNAL_MODE,
    ma_short=config.MA_SHORT,
    ma_medium=config.MA_MEDIUM,
    ma_long=config.MA_LONG,
)
sig_pdf = signal_df.to_pandas()
sig_pdf["datetime"] = pd.to_datetime(sig_pdf["datetime"])
merged = pdf.merge(sig_pdf[["datetime", "signal", "ma_s", "ma_m", "ma_l"]], on="datetime", how="left")

vc = sig_pdf["signal"].value_counts().sort_index()
print("信号分布（-1空 / 0震荡 / +1多）：")
for k, v in vc.items():
    print(f"  {int(k):+2d}  ->  {v} 根")

# TEST 段可视化
t0, t1 = pd.Timestamp(config.TEST_PERIOD[0]), pd.Timestamp(config.TEST_PERIOD[1])
sub = merged[(merged["datetime"] >= t0) & (merged["datetime"] <= t1)].copy()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(sub["datetime"], sub["close"], color="#1e293b", lw=1.2, label="收盘")
ax.plot(sub["datetime"], sub["ma_s"], color="#ef4444", lw=1, label=f"MA{config.MA_SHORT}")
ax.plot(sub["datetime"], sub["ma_m"], color="#f59e0b", lw=1, label=f"MA{config.MA_MEDIUM}")
ax.plot(sub["datetime"], sub["ma_l"], color="#22c55e", lw=1, label=f"MA{config.MA_LONG}")

long_m = sub["signal"] > 0
short_m = sub["signal"] < 0
ax.scatter(sub.loc[long_m, "datetime"], sub.loc[long_m, "close"], c="#16a34a", s=12, alpha=0.6, label="多头信号", zorder=3)
ax.scatter(sub.loc[short_m, "datetime"], sub.loc[short_m, "close"], c="#dc2626", s=12, alpha=0.6, label="空头信号", zorder=3)
ax.set_title(f"TEST 段三均线 {config.MA_SHORT}/{config.MA_MEDIUM}/{config.MA_LONG}")
ax.legend(loc="upper left", ncol=3)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3️⃣ 样本外回测 🏆

T 日收盘出信号 → T+1 成交；手数 = `floor(权益 × 95% / (现价×20×14%))`，封顶 1 手。

In [ ]:
from datetime import datetime
from vnpy.alpha import AlphaLab
from vnpy.alpha.strategy import BacktestingEngine
from vnpy.trader.constant import Interval

def to_dt(s):
    return datetime.strptime(s, "%Y-%m-%d")

lab = AlphaLab(str(config.LAB_PATH))
lab.save_bar_data(data.build_bar_data())
lab.add_contract_setting(
    config.VT_SYMBOL, config.LONG_RATE, config.SHORT_RATE,
    config.CONTRACT_SIZE, config.PRICE_TICK,
)

engine = BacktestingEngine(lab)
engine.set_parameters(
    vt_symbols=[config.VT_SYMBOL],
    interval=Interval.DAILY,
    start=to_dt(config.TEST_PERIOD[0]),
    end=to_dt(config.TEST_PERIOD[1]),
    capital=config.CAPITAL,
    annual_days=240,
)
engine.add_strategy(
    ShortTermStrategy,
    {
        "signal_threshold": config.SIGNAL_THRESHOLD,
        "position_pct": config.POSITION_PCT,
        "margin_rate": config.MARGIN_RATE,
        "max_lots": config.MAX_LOTS,
        "price_add_ticks": config.PRICE_ADD_TICKS,
    },
    signal_df,
)
engine.load_data()
engine.run_backtesting()
daily = engine.calculate_result()
stats = engine.calculate_statistics()

end_bal = float(stats.get("end_balance", 0) or 0)
total_ret = float(stats.get("total_return", 0) or 0)
print(f"TEST {config.TEST_PERIOD[0]} ~ {config.TEST_PERIOD[1]}")
print(f"  {config.CAPITAL:,} 元 -> {end_bal:,.0f} 元  总收益 {total_ret:.2f}%")
for k in ("annual_return", "max_ddpercent", "sharpe_ratio", "total_trade_count", "total_commission"):
    if k in stats:
        v = stats[k]
        print(f"  {k}: {float(v):.2f}" if isinstance(v, (int, float, str)) and k != "total_trade_count" else f"  {k}: {v}")

## 4️⃣ 绩效图表 📊

In [ ]:
if daily is not None and not daily.is_empty():
    ddf = daily.to_pandas()
    ddf["date"] = pd.to_datetime(ddf["date"])
    equity = config.CAPITAL + ddf["net_pnl"].cumsum()

    fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True, gridspec_kw={"height_ratios": [2, 1]})

    ax = axes[0]
    ax.plot(ddf["date"], equity, color="#2563eb", lw=1.5)
    ax.axhline(config.CAPITAL, color="#94a3b8", ls="--", lw=1)
    ax.fill_between(ddf["date"], config.CAPITAL, equity, where=equity >= config.CAPITAL, alpha=0.15, color="#22c55e")
    ax.fill_between(ddf["date"], config.CAPITAL, equity, where=equity < config.CAPITAL, alpha=0.15, color="#ef4444")
    ax.set_title(f"权益曲线（起始 {config.CAPITAL:,} 元）")
    ax.set_ylabel("权益（元）")
    ax.grid(alpha=0.3)

    ax2 = axes[1]
    colors = ["#22c55e" if x >= 0 else "#ef4444" for x in ddf["net_pnl"]]
    ax2.bar(ddf["date"], ddf["net_pnl"], color=colors, width=1.0, alpha=0.8)
    ax2.set_ylabel("日盈亏")
    ax2.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    tag = f"ma{config.MA_SHORT}_{config.MA_MEDIUM}_{config.MA_LONG}"
    out_path = config.ARTIFACT_PATH / f"short_{tag}_daily.csv"
    daily.write_csv(out_path)
    print(f"日度结果已保存: {out_path}")
else:
    print("无日度回测结果")

## 5️⃣ 关键指标一览

In [ ]:
from IPython.display import display

rows = []
for k, label in [
    ("start_date", "起始日"), ("end_date", "结束日"),
    ("total_return", "总收益率 %"), ("annual_return", "年化 %"),
    ("max_ddpercent", "最大回撤 %"), ("sharpe_ratio", "Sharpe"),
    ("return_drawdown_ratio", "收益回撤比"),
    ("total_trade_count", "成交笔数"), ("total_commission", "总手续费"),
    ("total_net_pnl", "净盈亏"),
]:
    if k in stats:
        v = stats[k]
        if k in ("total_return", "annual_return", "max_ddpercent"):
            v = f"{float(v):.2f}"
        elif k in ("sharpe_ratio", "return_drawdown_ratio"):
            v = f"{float(v):.2f}"
        rows.append({"指标": label, "数值": v})

summary = pd.DataFrame(rows)
display(summary)